# Trace Count v0 Colab

This notebook runs the full debug loop: generate data, train a tiny GPT-2 from scratch, evaluate teacher-forced and autoregressive metrics, run hidden-state probes, and plot results.

In [ ]:
# If this notebook is opened outside the cloned repo, set your GitHub URL here.
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"

from pathlib import Path
if not Path("pyproject.toml").exists():
    !git clone {REPO_URL} trace-counting
    %cd trace-counting

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
!pytest -q

In [ ]:
!python scripts/run_pipeline.py --config configs/experiment/debug.yaml --stage all

In [ ]:
from pathlib import Path
import pandas as pd

run_dir = Path("runs/trace_count_v0_debug/tiny_debug/completion_final_weighted_fw10_seed0")
log = pd.read_json(run_dir / "train_log.jsonl", lines=True)
log.tail()

In [ ]:
import json

with open(run_dir / "eval" / "summary_metrics.json") as f:
    metrics = json.load(f)
for split, value in metrics.items():
    tf = value.get("teacher_forced", {})
    ar = value.get("autoregressive", {})
    print(split, "tf_acc=", tf.get("count_accuracy"), "ar_acc=", ar.get("count_accuracy"), "trace_exact=", ar.get("trace_exact_match"))

In [ ]:
probe_summary = pd.read_csv(run_dir / "probes" / "probe_summary.csv")
probe_summary.sort_values(["target", "probe_type", "anchor", "layer"]).head(20)

In [ ]:
from IPython.display import Image, display

for path in sorted((run_dir / "plots").glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))

For the full v0 sweep, first generate `data/trace_count_v0`, then run:

```bash
python scripts/run_loss_mask_sweep.py --data_dir data/trace_count_v0 --model_config configs/model/small_main.yaml --model_name small_main --seeds 0,1,2 --max_steps 50000
python -m trace_counting.summarize --runs_dir runs/trace_count_v0 --out_csv runs/trace_count_v0/summary.csv --print_markdown
```